# Model report -- granular varieties, comparative advantage at the attraction-area level

The FIT of the structural model to its targets: the moment table, the sourcing-share and
downstream-sales fit, the extensive margin and the zero-supplier share with their standard
errors, the count curve, the Jacobian and what it says about identification, the moment
covariance matrices, and the untargeted distance elasticity of spatial comovement.

The companion notebook, `tests_counterfactuals.ipynb`, interrogates the same model one
test at a time and runs the counterfactuals.

Every function called here lives in `utils.py` or `report_lib.py`. Each section runs on
its own after the imports and the Constants cell.


## Setup

In [ ]:
%pip install matplotlib pandas statsmodels seaborn geopandas pyfixest pyarrow scipy


In [ ]:
# The libraries. Every function this notebook calls lives in one of them, so a section
# is edited in a file rather than in a 1,400-line cell, and the gates under `test/`
# import the same code instead of slicing it out of the notebook's JSON.
#
#   utils.py       the loader, `theta+`, the Ricardian geometry, and THE ECONOMY
#   report_lib.py  the moment fit, the count curve, the Jacobian, identification, and
#                  the untargeted distance elasticity
#
# Reload after editing one of them: `import importlib; importlib.reload(utils)` --
# or restart the kernel, which is safer once several modules have changed.
import os
import re
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

import utils
import report_lib
from utils import *
from report_lib import *


In [ ]:
# FIGURE STYLE -- one call, because the style itself lives in `utils.py`.
#
# It is deliberately NOT an rcParams block in this cell: a block here shadows the
# module's after import, so the two drift and a figure's size then depends on which
# cell was run last. Both notebooks make this same call and nothing else.
#
# `font_size` is the BODY size. It drives the axis labels, the ticks and the legend,
# and every figure-local size in the libraries is expressed RELATIVE to it, so the
# whole family scales together instead of leaving the annotations frozen at 8pt while
# the axis labels jump. A categorical NAME axis -- the per-buyer figures, some twenty
# commuting-zone names stacked in a few inches -- is additionally capped by the room
# each row actually has, so raising the body size no longer makes those labels collide
# or swamp the marks they annotate.
#
# `usetex=True` needs a LaTeX installation (latex + dvipng, and type1cm from
# texlive-latex-extra). It is off by default inside `utils` so the test gates never
# depend on one, and it is asked for here. The preamble adds T1 fontenc for the
# French commuting-zone names in the tick labels -- real accented glyphs rather than
# composed ones; on TeX Live 2018 and later they compile either way.
set_paper_style(font_size=20, usetex=True)

# Unused by any library function; kept because ad-hoc cells reach for it.
cmap = plt.get_cmap("viridis")

## Constants

In [ ]:
# mu = 1 -> step1 best parameters (theta_hat_1)
# mu = 2 -> step3 best parameters (theta_hat_2, the efficient estimate)
MU = 2

# Joint reporting: both industries in one table, one figure set each.
INDUSTRIES = [{"industry": "auto", "display_name": "Motor Vehicles"},
              {"industry": "aero", "display_name": "Aerospace"}]

# The run tree, and WHICH PARTS of it to read. This notebook reports the fit, so it
# reads the moment vector, the inference artefacts, the Jacobian and the count curve.
# It does NOT read `suppliers.parquet`: the firm-level panel behind the untargeted
# moment is simulated from `theta+` like every other economy (see the section below).
RUN_KWARGS = dict(base="..", profile_T=True, ca_level="aa",
                  granular=True, relax_n_lo=True, optimizer="pso",
                  parts=("core", "counts", "moments", "inference", "jacobian",
                         "geography"))

# Highest K kept anywhere in the count-curve reporting: the panels, the empirical
# increments read out of G_K.csv, and the bootstrap variances read out of G_K_var.csv
# are ALL truncated here. K = 0 is the targeted moment (block 6); everything above it
# is a free check on the SHAPE of the supplier-count distribution (gate V8).
COUNT_CURVE_K_MAX = 3
COUNT_CURVE_K = tuple(range(COUNT_CURVE_K_MAX + 1))

# Mean of log Dist in the EMPIRICAL estimation sample of the spatial-comovement
# regression (both industries). The reduced form is linear in the level of the
# exposure, so its delta/gamma is a compressed reading of an elasticity and the
# compression depends on this number.
EMPIRICAL_MEAN_LOG_D = 5.8

for _m in (utils, report_lib):
    for _k in ("COUNT_CURVE_K_MAX", "EMPIRICAL_MEAN_LOG_D"):
        if hasattr(_m, _k):
            setattr(_m, _k, globals()[_k])

out_folder = "../reporting_combined"
os.makedirs(out_folder, exist_ok=True)


## Loading

`utils.load_granular_data(industry, mu, parts=...)` rebuilds, on the Python side, the
layout `load_parameters.jl` builds under `--granular=true --ca_level=aa`: the FULL moment
vector

    [labor(1) | industry(S) | pi_r(R_d) | reg_coef(N_REG) | gamma_aa(S x n_AA, s-major)
     | Gbar_s(0)(S)]

and MOMENT_MASK exactly as the estimator builds it, plus both Jacobian axes and the
inference artefacts. `parts` says which blocks to read; this notebook asks for everything
but `firm`, since the firm-level panel behind the untargeted moment is simulated from
`theta+` rather than read from `suppliers.parquet`.

`mu` selects the estimate: `mu=1` reads `step1/` with standard errors from
`step2/inference/`, `mu=2` reads `step3/` for both.


# Reporting

## Moment table

Empirical against simulated for the two blocks that read naturally as a table: the
aggregate labor share (Panel A) and the sectoral industry shares (Panel B), both
industries side by side, with `---` where a sector is absent from an industry's sample.

The distance coefficients and the zero-supplier shares are *not* table panels here. They
are few enough to plot, and they are the two blocks where the sampling uncertainty
matters for reading the fit, so they get their own figures with standard errors below.


In [ ]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
tex = generate_combined_table(INDUSTRIES, mu=MU, **RUN_KWARGS,
                              output_file=os.path.join(out_folder, f"moments_comparison_combined_mu{MU}.tex"))
print(tex)


## Fit: sourcing shares and downstream sales

Simulated against empirical, on the 45-degree line. Bubble area is proportional to the
empirical value, so a visually large miss on a small moment is seen for what it is.

For the sourcing shares $\gamma_{s,a}$ the figure reproduces the first panel of the Julia
dashboard. Each sector's **reference area** is dropped from the criterion, so its
simulated value is not fitted directly; it is reconstructed from the within-sector
adding-up constraint

$$\gamma_{\text{ref},s} \;=\; c_s - \sum_{a \neq \text{ref}} \gamma_{s,a},$$

and drawn in a separate colour. The fit statistics annotated on the panel (a weighted
regression through the origin) use the non-reference points only — the reconstructed
ones are an identity, not an independent observation.


In [ ]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
for cfg in INDUSTRIES:
    industry = cfg["industry"]
    data = load_granular_data(industry, mu=MU, **RUN_KWARGS)
    globals().update(data)   # the analysis.ipynb pattern: binds input_folder, folder,
                             # coefs, emp_gamma_ls, X_dr, best_params, france, ... by name
    print(f"{industry}: S={data['S']}  n_AA={data['n_AA']}  N_REG={data['n_coef']}  "
          f"moments from {data['folder'].name}/{data['step_dir']}, "
          f"SEs from {data['inference_step']}/inference")

    plot_gamma_aa(data, save_to=f"{out_folder}/emp_sim_gamma_aa_{industry}_mu{MU}.pdf")
    plt.show()
    plot_pi_r(data, save_to=f"{out_folder}/emp_sim_pi_{industry}_mu{MU}.pdf")
    plt.show()


## Fit with standard errors: extensive margin and zero-supplier share

The two blocks where the comparison is only meaningful against sampling uncertainty: the
cloglog distance-bin coefficients (block 4) and the zero-supplier shares $\bar G_s(0)$
(block 6). Each tick carries the empirical value on the left and the simulated one on the
right, both with 95% bars. **Two different objects, deliberately:**

* **Empirical bars** — $\sqrt{\mathrm{diag}(\Sigma_{data})}$, the bootstrap sampling
  standard error of the moment *in the data*. How precisely the target is measured.
* **Simulated bars** — `se_moments_fitted.npy`, the standard error of the *fitted*
  moment, $\sqrt{\mathrm{diag}(G\,V\,G')}$, i.e. the estimation uncertainty of the
  parameters carried through to the moment they generate.

Under the profiled estimator (`--profile_T=true`, what these runs use) $V$ is the
**sandwich** covariance of $\hat\alpha$ and $G = \partial m / \partial \alpha$ is the
*profiled* Jacobian — $\alpha$ moved with $T$ following through the Sinkhorn inversion.
So the simulated bar is the delta-method propagation of $\hat\alpha$'s sandwich standard
error onto that moment, not an efficient standard error built from the free-parameter
Jacobian: under profiling $T$ is not a free parameter, it is a function of $\alpha$ and of
the sourcing-share targets, and the reported bar takes that dependence into account.
Two consequences worth keeping in mind when reading the figure: the bar does *not*
include the data noise of the $\gamma$ targets that $T$ is inverted from (that channel is
reported separately, in Julia's `inference_delta.txt`), and $\bar G_s(0)$'s bar does not
include the uncertainty of $\hat N_s$, which is calibrated on that very moment.

Both numbers come out of the Julia inference step; nothing is re-estimated here.

Each sector of the $\bar G_s(0)$ panel is annotated with its profiled variety count
$\hat N_s$, starred when it sits on a bound: a sector clamped at a bound cannot close its
own count residual through $\hat N_s$, so a visible miss there is expected rather than a
fit failure.

**Where the same numbers appear on the Julia side, and why they must agree.**
$\bar G_s(0)$ is written down in four places, and the panel shows two of them: the red
marker is `G_K.csv` at $K = 0$ (block 6 of the empirical moment vector, printed as
`G0_emp` in a stage `report.txt`), the blue one is block 6 of
`<step>/best_simulated_moments.npy`, which `run_reporting` evaluates at the **last stage
folder's** `best_params.npy` on `U_DRAWS`. The other two are `G0_target` and `G0_fit` in
`<step>/granular_diagnostics.npz` (and its `.txt` twin), which `report_granular` computes
at the $\hat\theta$ that step saved. The two $\theta$ are the same vector —
`run_optimization` returns its final sub-stage's parameters and writes that same vector
into the last stage folder — and $\bar G_s(0)$ is a closed form in the win counts, so it
carries no simulation noise of its own: the two pairs must agree to the solver tolerance.

**The rest of the curve, $\bar G_s(1), \bar G_s(2), \bar G_s(3)$.** Only $K = 0$ is
targeted, and it is targeted *tightly*: $\hat N_s$ is the integer that solves
$\bar G_s(0) = \hat G_s$, so a good fit in that panel is close to mechanical. Nothing in
the criterion asks the model to reproduce the **shape** of the supplier-count
distribution, which makes $K \geq 1$ a free check — gate V8 of
`documentation/granular_validation.md` — on whether one variety count per sector is
enough structure to generate the observed spread of $K_{ls}$, or whether the model
concentrates suppliers into too few cells (the simulated curve then climbs too fast) or
scatters them over too many (too slow).

Neither side of it is on disk. The empirical curve is the rest of `G_K.csv`, which
`load_parameters.jl` reads only at $K = 0$; the simulated curve Julia never evaluates at
all, since the moment vector stops there. `count_curve` rebuilds it from the two
diagnostics that determine it, $\hat q_{ls}$ and $\hat N_s$: a cell hosts a supplier for
each variety it wins somewhere, so $K_{ls} \sim \mathrm{Bin}(\hat N_s, \hat q_{ls})$ and
$\bar G_s(K) = \mathrm{mean}_l \Pr(K_{ls} \leq K)$. It is estimated under the same
unbiased convention `gbar_cell` uses — take the $\hat N_s$ varieties to be that many of
the $m$ simulated draws sampled without replacement, so the win count among them is
hypergeometric and $\mathbb{E}[C(k,j)C(m-k,\hat N_s-j)/C(m,\hat N_s)]$ is the binomial
pmf exactly — which at $K = 0$ collapses to `gbar_cell` itself. That is what ties the
untargeted panels to the targeted one, and it is checked rather than asserted: the
rebuilt $K = 0$ column must reproduce block 6 of the moment vector, or the function falls
back to the plug-in binomial and says so.

**Standard errors on $K = 0$ only.** $\Sigma_{data}$'s count block is the $S$ rows of
$\bar G_s(0)$; the bootstrap never covered the rest of the curve, so a bar there would be
invented.

They come apart for one reason, and it is a bookkeeping one rather than a modelling one:
`report_granular` writes the npz only when *its* step runs (Step 2 at $\hat\theta_1$,
Step 4 at $\hat\theta_2$), whereas `main.jl`'s post-hoc block rewrites
`best_simulated_moments.npy` on **every** invocation. A run resumed with
`run_step4 = false` therefore leaves the reporting frozen at an older $\theta$ while the
figure follows the newest one. `g0_consistency(data)` puts all four columns side by side
and says which of the two gaps opened; `plot_G0` calls it and warns if either does.


In [ ]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
for cfg in INDUSTRIES:
    industry = cfg["industry"]
    data = load_granular_data(industry, mu=MU, **RUN_KWARGS)
    globals().update(data)   # the analysis.ipynb pattern: binds input_folder, folder,
                             # coefs, emp_gamma_ls, X_dr, best_params, france, ... by name
    print(f"{industry}: S={data['S']}  n_AA={data['n_AA']}  N_REG={data['n_coef']}  "
          f"moments from {data['folder'].name}/{data['step_dir']}, "
          f"SEs from {data['inference_step']}/inference")

    plot_reg_coef(data, save_to=f"{out_folder}/emp_sim_reg_coef_{industry}_mu{MU}.pdf")
    plt.show()
    plot_G0(data, save_to=f"{out_folder}/emp_sim_G0_{industry}_mu{MU}.pdf")
    plt.show()
    # The figure against Julia's own reporting of the same moment, sector by sector.
    display(g0_consistency(data))
    # ... and the rest of the count curve, which nothing in the criterion targets.
    cc = count_curve(data, K_values=COUNT_CURVE_K)
    # one figure per K, each in its own file; K = 0 is `plot_G0` above, which carries
    # the N_hat annotation, so only the untargeted K >= 1 are drawn here
    plot_count_curve(data, df=cc, K_values=[K for K in COUNT_CURVE_K if K > 0],
                     save_to=f"{out_folder}/emp_sim_count_curve_{industry}_mu{MU}.pdf")
    plt.show()
    display(cc.unstack("K"))


# Jacobian and variance-covariance

## Jacobian — which parameter moves which moment

Rows are the moments the criterion actually uses (the masked vector, six blocks),
columns are the parameters. The parameter axis is
$[\Omega^L \mid \Omega^s \mid A \mid \alpha \mid T \mid N_s]$: the head parameters, the
trade-cost elasticity, the comparative advantages, and — appended on the right — the
**variety counts**.

**Why $N_s$ is in the picture.** $T$ and $N_s$ are both *calibrated* rather than
searched over: $T$ is the Sinkhorn image that reproduces the observed sourcing shares at
a given $\alpha$, $N_s$ is the solution of $\bar G_s(n) = \hat G_s$ found by integer
bisection. $T$ appeared in the Jacobian only because it happens to be a coordinate of the
parameter vector and $N_s$ is not — a bookkeeping accident, not a statement about the
model. The Julia side now appends $\partial m/\partial N_s$ to the saved Jacobian, so the
matrix covers every parameter the model has. Two conventions for reading those columns:

* the derivative is a unit **first difference** $m(N_s+1) - m(N_s)$. $N_s$ is an integer,
  so one variety *is* the step and there is no step size to choose;
* the zeros outside block 6 are **structural**, not numerical. The two evaluations share
  the same productivity draws, so any moment with no $N_s$ term differences to exactly
  `0.0`, and Julia asserts it (the extensive margin is $N_s$-free by Proposition 1, the
  win probabilities by Lemma 2). An exact zero in an $N_s$ column is the model saying the
  channel does not exist, not a small number that fell below a threshold.

**What is plotted is the elasticity** $\varepsilon_{jk} = \partial \log m_j / \partial
\log \theta_k$ (for $N_s$, $\hat N_s \cdot \partial m_j/\partial N_s / m_j$). The raw
$\partial m/\partial\theta$ mixes units across blocks and cannot share one colour scale.

**The noise-to-signal triptych.** These derivatives are finite differences of *simulated*
moments, so each entry is itself an estimate: `compute_jacobian` averages over $K$
independent draw sets and reports the across-replication standard deviation
$\sigma_{jk}$. An entry is readable only if $\sigma_{jk}/|\varepsilon_{jk}|$ is small —
a large elasticity with a larger standard deviation is not evidence of a channel, and a
zero there means "not measured", not "not there". So each elasticity figure comes in
three panels:

1. the elasticity Jacobian as it is;
2. the noise-to-signal ratio $\sigma/|\varepsilon|$ alone, with the cells that fail the
   criterion marked — which parts of panel 1 are readable at all;
3. the elasticity Jacobian with those cells blanked to white — what survives.

A cell fails when $\sigma/|\varepsilon| \geq$ `NOISE_MAX` (default 0.5: the Monte-Carlo
standard deviation is at least half the elasticity itself). Entries that are *exactly*
zero are treated as readable, since a structural zero carries no simulation noise —
distinguishing them from unmeasured entries is the whole point of the panel.

**Under `profile_T` this is the FREE-parameter Jacobian**, saved for diagnostics: it
answers "what would move if $T$ were free". The inference itself runs on the $\alpha$-only
profiled Jacobian, where $T$ follows $\alpha$ through the Sinkhorn inversion.


In [ ]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
for cfg in INDUSTRIES:
    industry = cfg["industry"]
    data = load_granular_data(industry, mu=MU, **RUN_KWARGS)
    globals().update(data)
    print(f"{industry}: S={data['S']}  n_AA={data['n_AA']}  N_REG={data['n_coef']}  "
          f"{data['n_N_cols']} variety-count columns in the Jacobian")

    # the triptych: as is, the noise-to-signal map, and the purged matrix
    plot_jacobian_triptych(data, noise_max=NOISE_MAX, out_folder=out_folder,
                           tag="jacobian")
    plot_jacobian_blocks(data, save_to=f"{out_folder}/jacobian_blocks_{industry}_mu{MU}.pdf")
    plt.show()
    display(jacobian_block_summary(data, noise_max=NOISE_MAX))


## Variance-covariance of the moments


Three matrices, all written by Step 2:

* $\Sigma_{data}$ — the bootstrap covariance of the **empirical** moments: how precisely
  the targets are measured;
* $\Sigma_{sim}$ — the covariance across $K$ re-simulations at $\hat\theta_1$: how much
  of the moment vector is Monte-Carlo noise from the simulator;
* $\Omega = \Sigma_{data} + \Sigma_{sim}$ — what the estimator actually weighted with,
  $W = \Omega^{-1}$.

Two views of each. The **covariance** panels are on a common diverging scale, so the
relative magnitude of the three is visible — which is the point, since
$\mathrm{tr}(\Sigma_{sim})/\mathrm{tr}(\Sigma_{data})$ is the share of the weighting that
is simulation rather than data, and the dial for it is `--n_rho_inf`. The **correlation**
panels rescale each matrix by its own diagonal, which removes that magnitude and leaves
the dependence structure: whether the sampling error of the sourcing shares is correlated
across areas, whether the simulation noise is (it is, through the shared draws), and
whether $\Omega$ inherits its correlation from the data or from the simulator.


In [ ]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
for cfg in INDUSTRIES:
    industry = cfg["industry"]
    data = load_granular_data(industry, mu=MU, **RUN_KWARGS)
    globals().update(data)

    plot_variance_covariance(data, save_to=f"{out_folder}/varcov_{industry}.pdf")
    plt.show()
    plot_moment_correlation(
        data, save_to=f"{out_folder}/moment_correlation_{industry}.pdf")
    plt.show()
    display(variance_covariance_summary(data))


## Identification / sensitivity

A sensitivity analysis fixes every parameter, moves one of them above and below its
estimate, and checks that the moments respond smoothly and in the direction the design
predicts. The elasticity Jacobian *is* that experiment, linearised at the estimate:
$\varepsilon_{jk} = \partial \log m_j / \partial \log \theta_k$ is the local slope of
moment $j$ against parameter $k$. With ten sectors, the attraction areas, the $T$ block
and the variety counts there are far too many parameters to draw a curve each, so the
matrix is drawn instead and everything too small — or too noisy — is blanked out.

Two thresholds, and they answer different questions:

* $|\varepsilon| \geq$ `IDENT_THRESHOLD` (0.01): a 1% move in the parameter shifts the
  moment by at least 0.01%. **Is there a channel?**
* $\sigma/|\varepsilon| <$ `NOISE_MAX` (0.5): the elasticity is estimated precisely
  enough relative to its own size. **Can we see it?** An entry that fails this is not
  evidence of a weak channel; it is evidence of nothing at all, and the summary counts it
  separately rather than pooling it with the structural zeros.

What the figures are meant to show: the three channels are close to orthogonal *by
construction*. $\alpha$ is read from a **within**-area distance gradient that comparative
advantage cannot touch, since comparative advantage is constant within an area; $T_{as}$
is read from **across**-area value shares; $N_s$ is read from the **level** of the
empty-region share, which the sector $\times$ nearest-downstream fixed effect strips out
of the $\alpha$ regression. So the expected picture is near-block-diagonal on the
(Trade cost, Comparative advantage, Variety count) $\times$ (Extensive margin, Regional
sourcing shares, Zero-supplier share) corner — and in the $N_s$ columns *exact* zeros
rather than merely small numbers.


In [ ]:
# --- RUN THIS SECTION ON ITS OWN -------------------------------------------
for cfg in INDUSTRIES:
    industry = cfg["industry"]
    data = load_granular_data(industry, mu=MU, **RUN_KWARGS)
    globals().update(data)

    plot_identification_map(
        data, threshold=IDENT_THRESHOLD, noise_max=NOISE_MAX,
        save_to=f"{out_folder}/identification_map_{industry}_mu{MU}.pdf")
    plt.show()
    plot_channel_elasticities(
        data, threshold=IDENT_THRESHOLD, noise_max=NOISE_MAX,
        save_to=f"{out_folder}/identification_channels_{industry}_mu{MU}.pdf")
    plt.show()
    plot_jacobian_thresholded(
        data, threshold=IDENT_THRESHOLD, noise_max=NOISE_MAX,
        save_to=f"{out_folder}/jacobian_thresholded_{industry}_mu{MU}.pdf")
    plt.show()
    display(identification_summary(data, threshold=IDENT_THRESHOLD, noise_max=NOISE_MAX))
